## 01 Duckduckgo Search Tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# LLMs have a strict knowledge cutoff (e.g., they don't know the news from today).
# We give them an Internet Search Tool so they can fetch live data before answering.
search_tool = DuckDuckGoSearchRun()

# You can test the tool manually:
# print(search_tool.invoke("Who won the Super Bowl in 2024?"))


## 02 Agent With Tool

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatGroq(model='llama-3.3-70b-versatile')
tools = [search_tool]

# The Prompt must contain a placeholder for the agent's scratchpad (where it thinks)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools if you need to."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Agent and Executor
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# The agent will realize it needs to search the internet to answer this live question
# agent_executor.invoke({"input": "What is the current stock price of Apple?"})


## 03 Agent With Middleware

In [ ]:
# In complex agents, the context window can fill up quickly as the agent uses tools and talks.
# We can use Middleware (like SummarizationMiddleware) to intercept the agent's memory 
# and summarize it automatically if it exceeds a certain token count.

"""
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-4o",
    tools=[search_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 4000), # If conversation hits 4000 tokens
            keep=("messages", 20),    # Keep the last 20 messages, summarize the rest
        ),
    ],
)
"""
